In [4]:
# 重命名image rename
import os


root = "E:/TrainingFramework/DataSets/Dataset10086_boge_void_2nn"
image_folder = "imagesTr"

id_name = "boge_void_2nn"

path1 = os.path.join(root, image_folder)
for i, filename in enumerate(os.listdir(path1)):
    os.rename(os.path.join(path1, filename), os.path.join(path1, id_name + "_" + str(i) + "_0000.png"))

In [5]:
# 重命名label rename
import os


root = "E:/TrainingFramework/DataSets/Dataset10086_boge_void_2nn"
label_folder = "labelsTr"

id_name = "boge_void_2nn"

path2 = os.path.join(root, label_folder)
for i, filename in enumerate(os.listdir(path2)):
    os.rename(os.path.join(path2, filename), os.path.join(path2, id_name + "_" + str(i) + ".png"))

In [1]:
# label2mask png
import cv2, os
import numpy as np
import json


label_path = "E:/TrainingFramework/nnUNet-master/DATASET/rrrr/Dataset003_Position/image/"
label_extension = ".json"

# 0 : bg
cls = {
    "a": 1,
    "b": 2,
    "c": 3,
}


# json -> mask image

for i, filename in enumerate(os.listdir(label_path)):
    file = os.path.join(label_path, filename)
    if file.endswith(label_extension):
        with open(file, "r") as f:
            data = json.load(f)
            # h,w
            h = data["imageHeight"]
            w = data["imageWidth"]

            # mask
            mask = np.zeros((h, w), dtype=np.uint8)

            # record
            areas = []
            points = []
            labels = []

            for shape in data["shapes"]:
                label = shape["label"]
                pts = shape["points"]
                area = cv2.contourArea(np.array(pts, dtype=np.float32))

                areas.append(area)
                points.append(pts)
                labels.append(label)

            areas = np.array(areas)
            index = np.argsort(areas)  # 从小到大
            index = index[::-1]  # 从大到小
            for idx in index:
                pts = np.array(points[idx], dtype=np.int32)
                mask = cv2.fillPoly(mask, [pts], cls[labels[idx]])

            # cv2.imshow('img', mask)
            # cv2.waitKey(0)
            ...
            crop = len(label_extension)
            cv2.imwrite(os.path.join(label_path, f"{filename[:-crop]}.png"), mask)

In [8]:
# 可视化mask图
import numpy as np
import os, cv2

path = "E:/TrainingFramework/nnUNet-master/DATASET/nnUNet_raw/Dataset10086_boge_void_2nn/labelsTr/"

for filename in os.listdir(path):
    if filename.endswith(".png"):
        img = cv2.imread(os.path.join(path, filename), cv2.IMREAD_GRAYSCALE)
        labels = np.unique(img)
        max_ = img.max()
        img = np.astype(img, np.float64)
        img = img * 255 / max_
        img = np.astype(img, np.uint8)
        cv2.imshow("img", img)
        cv2.waitKey(0)

KeyboardInterrupt: 